<a href="https://colab.research.google.com/github/nathanpaul423/NathanPaul-DataScience-Project/blob/main/S6E11%7CPredictCreditCardFraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [4]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

playground_series_s6e11_path = kagglehub.competition_download('playground-series-s6e11')

print('Data source import complete.')


100%|██████████| 29.5M/29.5M [00:00<00:00, 166MB/s]

Extracting files...


Data source import complete.


In [5]:
%reload_ext cudf.pandas

ModuleNotFoundError: No module named 'cudf'

In [6]:
pip install pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 15.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pytabkit]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import cudf

# kagglehub.dataset_download('<owner>/<dataset-slug>')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, KFold
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import xgboost as xgb

import lightgbm as lgb
from catboost import CatBoostClassifier


import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*Falling back to prediction using DMatrix due to mismatched devices.*",
    category=UserWarning,
    module=r"xgboost(\..*)?"
)


from pytabkit import RealMLP_TD_Classifier

In [ ]:
train = cudf.read_csv('/kaggle/input/competitions/playground-series-s6e11/train.csv')
test = cudf.read_csv('/kaggle/input/competitions/playground-series-s6e11/test.csv')
display(train.info())

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 26 columns):
 #   Column                        Non-Null Count   Dtype
---  ------                        --------------   -----
 0   id                            699960 non-null  int64
 1   amount_usd                    699960 non-null  float64
 2   hours_since_last_txn          699960 non-null  float64
 3   txn_count_last_24h            699960 non-null  int64
 4   distance_from_home_km         699960 non-null  float64
 5   card_age_months               699960 non-null  int64
 6   customer_age                  699960 non-null  int64
 7   account_balance_usd           699960 non-null  float64
 8   cvv_retry_count               699960 non-null  int64
 9   velocity_score                699960 non-null  float64
 10  time_of_day_hour              699960 non-null  int64
 11  day_of_week                   699960 non-null  int64
 12  merchant_risk_score           699960 non-null  float64
 13  

None

In [ ]:
TARGET = 'is_fraud'

In [ ]:
(train['is_fraud'].value_counts(normalize=True)*100).round(2)

is_fraud
0.0    98.33
1.0     1.67
Name: proportion, dtype: float64

In [ ]:
width = max([len(c) for c in train.columns])
print(f'{'column':{width}} UniqueValues')
for col in train.columns:
    print(f'{col:{width}} --> {train[col].nunique()}')

column                       UniqueValues
id                           --> 699960
amount_usd                   --> 13387
hours_since_last_txn         --> 3633
txn_count_last_24h           --> 12
distance_from_home_km        --> 4365
card_age_months              --> 49
customer_age                 --> 64
account_balance_usd          --> 43317
cvv_retry_count              --> 4
velocity_score               --> 666
time_of_day_hour             --> 24
day_of_week                  --> 7
merchant_risk_score          --> 950
prior_disputes               --> 5
merchant_category            --> 12
card_type                    --> 5
auth_method                  --> 5
channel                      --> 5
device_type                  --> 8
is_foreign_transaction       --> 2
is_new_merchant              --> 2
used_vpn                     --> 2
ip_country_mismatch          --> 2
billing_shipping_mismatch    --> 2
is_ai_generated_scam_attempt --> 2
is_fraud                     --> 2


In [ ]:
BASE_FEAT = [c for c in train.columns if c not in ['id','is_fraud']]
BOOL_COL = [c for c in train.select_dtypes('boolean') if c not in ['id','is_fraud']]
CAT_COL = [c for c in train.select_dtypes('object') if c not in ['id','is_fraud']]
NUM_COL = [c for c in train.select_dtypes('number') if c not in ['id','is_fraud']]

In [ ]:
missing = [c for c in train.columns if c not in BOOL_COL+CAT_COL+NUM_COL]
missing

['id', 'is_fraud']

In [ ]:
assert len(BOOL_COL+CAT_COL+NUM_COL) == len(BASE_FEAT)

In [ ]:
## Basic Split

In [ ]:
X = train[BASE_FEAT].copy()
X[CAT_COL+BOOL_COL] = X[CAT_COL+BOOL_COL].astype('category')
X[BOOL_COL] = X[BOOL_COL].astype(int)
y = train[TARGET].astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42, stratify=y)

## BASED RUN

In [ ]:
# model = XGBClassifier(**xgboost_params)

In [ ]:
# weights = np.where(y_train == 1, 3.0, 1.0)

In [ ]:
# model.fit(X_train,y_train, eval_set=[(X_test, y_test)], sample_weight=weights, verbose=False)

In [ ]:
# y_pred = model.predict_proba(X_test)[:,1]
# roc_auc_score(y_test, y_pred)

## StratifiedKFold

In [ ]:
#model parameters:

xgboost_params = {
                    'objective':"binary:logistic",
                    'max_depth':2,
                    'enable_categorical':True,
                    'eval_metric':"logloss",
                    'n_estimators':10000,
                    'learning_rate':0.01,
                    'subsample':0.8,
                    'colsample_bytree':0.8,
                    'tree_method':"hist",
                    'enable_categorical':True,
                    'early_stopping_rounds':200,
                    'device':'cuda'
}


In [ ]:
combined = pd.concat([train[BASE_FEAT],test[BASE_FEAT]], axis = 0 , ignore_index=True)
combined.head(3)

In [ ]:
combined[CAT_COL] = combined[CAT_COL].astype('category')
combined[BOOL_COL] = combined[BOOL_COL].astype('int')

In [ ]:
assert len(combined) == len(train)+len(test)

In [ ]:
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
oofs = np.zeros(len(X))
y_pred = np.zeros(len(test))
model = XGBClassifier(**xgboost_params)

for fold, (tr, va) in enumerate(skf.split(X,y)):
    model.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va],y[va])], verbose = 0)
    oofs[va] = model.predict_proba(X.iloc[va])[:,1]
    auc = roc_auc_score(y[va],oofs[va])

    y_pred = model.predict_proba(combined[len(train):])[:,1]/skf.n_splits

    print(f'{fold + 1} AUC: {auc:.4f}')

In [ ]:
pd.DataFrame({'id':test['id'], 'is_fraud':y_pred}).to_csv('submission.csv', index=False)

## GridSearch

In [ ]:
#model parameters:

xgboost_params = {
                    'objective':"binary:logistic",
                    'max_depth':2,
                    'enable_categorical':True,
                    'eval_metric':"logloss",
                    'n_estimators':10000,
                    'learning_rate':0.01,
                    'max_depth':4,
                    'subsample':0.8,
                    'colsample_bytree':0.8,
                    'tree_method':"hist",
                    'enable_categorical':True,
                    # 'early_stopping_rounds':200,
                    'device':'cuda'
}


In [ ]:
param_grid = {
    'max_depth':[2,3,4,5],
    'learning_rate':[0.01,0.02,0.03,0.04,0.05],
    'n_estimators':[10000,20000,30000,40000]
}

In [ ]:
model = XGBClassifier(**xgboost_params)

In [ ]:
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=1, verbose=0)
grid_search.fit(X_train, y_train)


In [ ]:
# pd.DataFrame(grid_search.cv_results_)

## Applied Hyperparameters

In [ ]:
# #model parameters:

# xgboost_params = {
#                     'objective':"binary:logistic",
#                     'max_depth':2,
#                     'enable_categorical':True,
#                     'eval_metric':"logloss",
#                     'n_estimators':10000,
#                     'learning_rate':0.01,
#                     'max_depth':4,
#                     'subsample':0.8,
#                     'colsample_bytree':0.8,
#                     'tree_method':"hist",
#                     'enable_categorical':True,
#                     'early_stopping_rounds':200,
#                     'device':'cuda'
# }


In [ ]:
# model = XGBClassifier(**xgboost_params)

In [ ]:
# model.fit(X_train,y_train, eval_set=[(X_test, y_test)], verbose=False)

In [ ]:
# y_pred = model.predict_proba(X_test)[:,1]
# roc_auc_score(y_test, y_pred)

## RealMLPCLassifier

In [ ]:
X = train[BASE_FEAT].copy()
X[CAT_COL+BOOL_COL] = X[CAT_COL+BOOL_COL].astype('category')
X[BOOL_COL] = X[BOOL_COL].astype(int)
y = train[TARGET].astype(int)

In [ ]:
cat_map = {}
CAT_COL_CODED = []
for i in CAT_COL:
    cat_map = {cat:i for i, cat in enumerate(X[i].drop_duplicates().to_arrow().to_pandas)}
    col_name = f'{i}_coded'
    X[col_name] = X[i].map(cat_map)
    CAT_COL_CODED.append(col_name)

TypeError: Series object is not iterable. Consider using `.to_arrow()`, `.to_pandas()` or `.values_host` if you wish to iterate over the values.

In [ ]:
X[CAT_COL_CODED] = X[CAT_COL_CODED].astype(int)
X_coded = X[BOOL_COL+NUM_COL+CAT_COL_CODED].copy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_coded,y, test_size = 0.2, random_state = 42, stratify=y)

In [ ]:
model = RealMLP_TD_Classifier(
    device="cuda",          # try this if the Lightning warning persists
    random_state=42,
    n_cv=1,
    n_refit=0,
    use_early_stopping=True,
    n_epochs=100,            # maximum; early stopping can stop sooner
    batch_size=256,
    hidden_sizes=[256] * 3,
    val_metric_name="cross_entropy",
    use_ls=False,
    lr=0.1,
    verbosity=2,
)

model.fit(
    X_train, y_train,
    X_test, y_test,
    cat_col_names=CAT_COL_CODED,
)

In [ ]:
model.predict_proba(X_test)

In [8]:
print('x')

x
